<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/Precipitation_re-analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install xee

In [ ]:
import ee
import xarray

In [ ]:
ee.Authenticate()
ee.Initialize(project = 'ee-grmntfrancis0',
              opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
ds = xarray.open_dataset('ee://ECMWF/ERA5_LAND/HOURLY', engine='ee')

In [ ]:
ds = xarray.open_dataset('ee://ECMWF/ERA5_LAND/HOURLY', engine='ee',
                         crs='EPSG:4326', scale=0.25)

In [ ]:
ic = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY').filterDate(
    '1992-10-05', '1993-03-31')
ds = xarray.open_dataset(ic, engine='ee', crs='EPSG:4326', scale=0.25)

In [ ]:
ic = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY').filterDate(
    '1992-10-05', '1993-03-31')
leg1 = ee.Geometry.Rectangle(113.33, -43.63, 153.56, -10.66)
ds = xarray.open_dataset(
    ic,
    engine='ee',
    projection=ic.first().select(0).projection(),
    geometry=leg1
)

In [ ]:
ic = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY').filterDate(
    '1992-10-05', '1993-03-31')
leg1 = ee.Geometry.Rectangle(113.33, -43.63, 153.56, -10.66)
ds = xarray.open_dataset(
    ic,
    engine='ee',
    projection=ic.first().select(0).projection(),
    geometry=leg1
)

In [ ]:
!pip install xarray pooch

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

xr.set_options(keep_attrs=True, display_expand_data=False)
np.set_printoptions(threshold=10, edgeitems=2)

In [ ]:
ds = xr.tutorial.open_dataset("air_temperature")
ds

In [ ]:
temperature = ds["air"]
temperature

In [ ]:
ds.air

In [ ]:
temperature.values

In [ ]:
temperature.dims

In [ ]:
temperature.coords

In [ ]:
temperature.attrs

In [ ]:
selected_data = temperature.sel(time="2013-01-01",  lat="40.0", lon="260.0")
selected_data

In [ ]:
time_slice = temperature.sel(time=slice("2013-01-01", "2013-01-31"))
time_slice

In [ ]:
mean_temperature = temperature.mean(dim="time")
mean_temperature

In [ ]:
anomalies = temperature - mean_temperature
anomalies

In [ ]:
mean_temperature.plot()
plt.show()

In [ ]:
mean_temperature.plot(cmap="jet", figsize=(10, 6))
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Mean Temperature")

In [ ]:
temperature.sel(lat=40.0, lon=260.0).plot()
plt.show()

In [ ]:
print(ds.data_vars)

In [ ]:
temperature = ds["air"]

In [ ]:
mean_temp_ds = ds.mean(dim="time")
mean_temp_ds

In [ ]:
lat = ds.air.lat.data
lon = ds.air.lon.data
temp = ds.air.data

In [ ]:
temp.shape

In [ ]:
plt.figure()
plt.pcolormesh(lon, lat, temp[0, :, :])

In [ ]:
ds.air.isel(time=0).plot(x="lon")

In [ ]:
ds.air.sel(time="2013-01-01T00:00:00").plot(x="lon")

In [ ]:
ds.sel(time="2013-05")

In [ ]:
ds.sel(time=slice("2013-05", "2013-07"))

In [ ]:
ds.air.isel(time=0, lat=2, lon=3)

In [ ]:
seasonal_mean = ds.groupby("time.season").mean()
seasonal_mean.air.plot(col="season")

In [ ]:
cell_area = xr.ones_like(ds.air.lon)  # Placeholder for actual area calculation
weighted_mean = ds.weighted(cell_area).mean(dim=["lon", "lat"])
weighted_mean.air.plot()

In [ ]:
ds.air.isel(time=0).rolling(lat=5, lon=5).mean().plot()

In [ ]:
plt.figure(figsize=(10, 6))
temperature = ds["air"].sel(lat=40.0, lon=260.0)
temperature.plot(label="Original")
smoothed_temperature = temperature.rolling(time=20, center=True).mean()
smoothed_temperature.plot(label="Smoothed")
plt.title("Temperature Time Series (lat=40.0, lon=260.0)")
plt.xlabel("Time")
plt.ylabel("Temperature (K)")
plt.legend()
plt.show()

In [ ]:
ds["air"] = ds["air"].astype("float32")
ds.to_netcdf("air_temperature.nc")

In [ ]:
loaded_data = xr.open_dataset("air_temperature.nc")
loaded_data